# 43 Izu with a time-dependent coupling depth

## Time-dependent implementation

### Preamble

Set some path information.

In [ ]:
import sys, os, shutil
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

Loading everything we need from `sz_problem` and also set our default plotting and output preferences.

In [ ]:
import fenics_sz.utils
from fenics_sz.sz_problems.sz_params import allsz_params, default_params
from fenics_sz.sz_problems.sz_slab import create_slab, plot_slab
from fenics_sz.sz_problems.sz_geometry import create_sz_geometry
from fenics_sz.applications.codillo_setup import TDCDDislSubductionProblem
import numpy as np
import dolfinx as df
import pyvista as pv
import pathlib
import copy
import matplotlib.pyplot as pl
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)

### Parameters

We first select the name and resolution scale, `resscale` and target Courant number `cfl` of the model.

```{admonition} Resolution
By default the resolution (both spatial and temporal) is low to allow for a quick runtime and smaller website size.  If sufficient computational resources are available set a lower `resscale` and a lower `cfl` to get higher spatial and temporal resolutions respectively. This is necessary to get results with sufficient accuracy for scientific interpretation.
```


In [ ]:
name = "43_Izu"
resscale = 3.0
cfl      = 3.0

Then load the remaining parameters from the global suite.

In [ ]:
szdict = allsz_params[name]
print("{}:".format(name))
print("{:<20} {:<10}".format('Key','Value'))
print("-"*85)
for k, v in allsz_params[name].items():
    if v is not None and k not in ['z0', 'z15']: print("{:<20} {}".format(k, v))

Modify the age of subduction to 52 Myr:

In [ ]:
szdict['As'] = 52.0

### Setup

Setup a slab.

In [ ]:
slab = create_slab(szdict['xs'], szdict['ys'], resscale, szdict['lc_depth'])
_ = plot_slab(slab)

Create the subduction zome geometry around the slab.

In [ ]:
geom = create_sz_geometry(slab, resscale, szdict['sztype'], szdict['io_depth'], szdict['extra_width'], 
                             szdict['coast_distance'], szdict['lc_depth'], szdict['uc_depth'])
_ = geom.plot()

Finally, declare the `TDDislSubductionProblem` problem class using the dictionary of parameters.

In [ ]:
sz = TDCDDislSubductionProblem(geom, **szdict, 
                               cd0=30, cdf=80, dcd=default_params['coupling_depth_range'],
                               tc0=0.0, tcf=szdict['As'])

### Solve

Solve using a dislocation creep rheology.

In [ ]:
# save period
save_period = 1.0

# Select the timestep based on the approximate target Courant number
dt = cfl*resscale/szdict['Vs']
# Reduce the timestep to get an integer number of timesteps per save period
dt = save_period/np.ceil(save_period/dt)

solutions = sz.solve(szdict['As'], dt, theta=0.5, rtol=1.e-1, verbosity=2, save_period=save_period)

### Plot

Plot the solution at the finish time.

In [ ]:
plotter = pv.Plotter()
fenics_sz.utils.plot.plot_scalar(sz.T_i, plotter=plotter, scale=sz.T0, gather=True, cmap='coolwarm', scalar_bar_args={'title': 'Temperature (deg C)', 'bold':True})
fenics_sz.utils.plot.plot_vector_glyphs(sz.vw_i, plotter=plotter, gather=True, factor=0.1, color='k', scale=fenics_sz.utils.mps_to_mmpyr(sz.v0))
fenics_sz.utils.plot.plot_vector_glyphs(sz.vs_i, plotter=plotter, gather=True, factor=0.1, color='k', scale=fenics_sz.utils.mps_to_mmpyr(sz.v0))
geom.pyvistaplot(plotter=plotter, color='green', width=2)
cdpt = slab.findpoint('Slab::FullCouplingDepth')
fenics_sz.utils.plot.plot_points([[cdpt.x, cdpt.y, 0.0]], plotter=plotter, render_points_as_spheres=True, point_size=10.0, color='green')
fenics_sz.utils.plot.plot_show(plotter)
fenics_sz.utils.plot.plot_save(plotter, output_folder / "{}_td_solution_resscale_{:.2f}_cfl_{:.2f}.png".format(name, resscale, cfl,))

Save it to disk so that it can be examined with other visualization software (e.g. [Paraview](https://www.paraview.org/)).

In [ ]:
filename = output_folder / "{}_td_solution_resscale_{:.2f}_cfl_{:.2f}.bp".format(name, resscale, cfl,)
with df.io.VTXWriter(sz.mesh.comm, filename, [sz.T_i, sz.vs_i, sz.vw_i]) as vtx:
    vtx.write(0.0)
# zip the .bp folder so that it can be downloaded from jupyter lab
shutil.make_archive(str(filename), 'zip', root_dir=str(filename.parent), base_dir=str(filename.name))

### Plot slab temperatures over time

In [ ]:
# set up a figure
fig, axs = pl.subplots(1,2)

for d, depth in enumerate([0.0, 7.0]):
    # get some points along the slab
    slab = copy.deepcopy(sz.geom.slab_spline)
    slab.translatenormalandcrop(-depth)
    slabpoints = np.array([[curve.points[0].x, curve.points[0].y, 0.0] for curve in slab.interpcurves])
    cinds, cells = fenics_sz.utils.mesh.get_cell_collisions(slabpoints, sz.mesh)

    for sol in solutions[::5]: # remove [::5] to get them all
        t = sol['t']
        T = sol['T']
        # plot the slab temperatures
        Tslab = T.eval(slabpoints, cells)[:,0]
        axs[d].plot(Tslab, -slabpoints[:,1], label='t = {:.0f} Myr'.format(t))

        mode = 'a'
        if d == 0: mode = 'w'
        with open(output_folder / 'slab_temperatures_{:.0f}.txt'.format(t), mode) as f:
            np.savetxt(f, 
                    np.column_stack((slabpoints[:,0], slabpoints[:,1], np.ones(len(slabpoints[:,1]))*depth, Tslab)), 
                    delimiter=' ')
    
    axs[d].set_xlabel('T ($^\circ$C)')
    axs[d].set_ylabel('z (km)')
    axs[d].set_title('{}km depth into slab'.format(depth,))
    axs[d].legend()
    axs[d].invert_yaxis()


In [ ]:
# intersection of crust and slab
crustpt = sz.geom.slab_spline.intersecty(-7.0)
# intersection of aribtrary x and slab
x = 200
arbpt = sz.geom.slab_spline.intersectx(x)
crustpt[0], crustpt[1], arbpt[0], arbpt[1]

In [ ]:
sz.geom.coast_distance, sz.deltaxcoast

In [ ]:
sz.geom.trench_y, sz.deltaztrench

In [ ]:
x = 135
np.minimum(np.maximum(sz.deltaztrench*(1.0 - x/max(sz.deltaxcoast, np.finfo(float).eps)), 0.0), sz.deltaztrench)